In [1]:
import os
%pwd

'c:\\Users\\singh\\OneDrive\\Desktop\\IMDB Projetc\\research'

In [ ]:
# os.chdir('../')
%pwd

'c:\\Users\\singh\\OneDrive\\Desktop\\IMDB Projetc'

In [3]:
import os
from dotenv import load_dotenv
from dataclasses import dataclass
load_dotenv()

True

In [4]:
from pathlib import Path

@dataclass
class DataIngestionConfig:
    root_dir : Path
    source_URL: Path
    local_data_file : Path
    unzip_dir : Path

In [5]:
from src.IMDBSentimentAnalysis.constants import *
from src.IMDBSentimentAnalysis.utils.main import read_yaml, create_directories

[ 2026-05-18 20:06:47,004 : INFO : __init__ : Logger initialized successfully ]


In [6]:
class ConfigurationManager:
    def __init__(self, config_filepath = CONFIG_FILE_PATH):
        self.config = read_yaml(config_filepath)
        create_directories([self.config.artifacts_root])


    def get_data_ingestion_config(self) -> DataIngestionConfig:
        config = self.config.data_ingestion
        create_directories([config.root_dir])

        data_ingestion_config = DataIngestionConfig(
            root_dir        = config.root_dir,
            source_URL      = config.source_URL,
            local_data_file = config.local_data_file,
            unzip_dir       = config.unzip_dir
        )

        return data_ingestion_config

In [7]:
import os
import zipfile
from pathlib import Path
from dotenv import load_dotenv
import kaggle
import logging

load_dotenv()

logger = logging.getLogger(__name__)


class DataIngestion:
    def __init__(self, config: DataIngestionConfig):
        self.config = config


    def set_kaggle_credentials(self):
        """Load Kaggle credentials from .env file."""
        try:
            os.environ['KAGGLE_USERNAME'] = os.getenv('KAGGLE_USERNAME')
            os.environ['KAGGLE_KEY']      = os.getenv('KAGGLE_KEY')

            kaggle.api.authenticate()
            logger.info("✅ Kaggle credentials authenticated successfully")

        except Exception as e:
            logger.error(f"❌ Kaggle authentication failed: {e}")
            raise e


    def download_dataset(self):
        """Download dataset from Kaggle using source URL in config."""
        try:
            # Extract dataset identifier from URL
            # URL format: https://www.kaggle.com/datasets/<owner>/<dataset-name>
            dataset_id = "/".join(self.config.source_URL.split("/")[-2:])

            logger.info(f"Downloading dataset: {dataset_id}")
            logger.info(f"Saving to         : {self.config.root_dir}")

            kaggle.api.dataset_download_files(
                dataset   = dataset_id,
                path      = self.config.root_dir,
                unzip     = False           # we handle unzip separately
            )

            logger.info("✅ Dataset downloaded successfully")

        except Exception as e:
            logger.error(f"❌ Dataset download failed: {e}")
            raise e


    def extract_zip(self):
        """Extract downloaded zip file to unzip_dir."""
        try:
            unzip_path = self.config.unzip_dir

            os.makedirs(unzip_path, exist_ok=True)

            # Find the downloaded zip file
            zip_file_path = None
            for file in os.listdir(self.config.root_dir):
                if file.endswith(".zip"):
                    zip_file_path = os.path.join(self.config.root_dir, file)
                    break

            if zip_file_path is None:
                raise FileNotFoundError("No zip file found in root_dir")

            with zipfile.ZipFile(zip_file_path, 'r') as zip_ref:
                zip_ref.extractall(unzip_path)

            logger.info(f"✅ Extracted zip to: {unzip_path}")

        except Exception as e:
            logger.error(f"❌ Extraction failed: {e}")
            raise e


    def verify_data(self):
        """Verify the downloaded CSV file exists and is non-empty."""
        try:
            file_path = Path(self.config.local_data_file)

            if not file_path.exists():
                raise FileNotFoundError(f"Expected file not found: {file_path}")

            size_mb = file_path.stat().st_size / (1024 * 1024)

            logger.info(f"✅ File verified    : {file_path}")
            logger.info(f"   File size        : {size_mb:.2f} MB")

        except Exception as e:
            logger.error(f"❌ Verification failed: {e}")
            raise e

In [8]:
object = ConfigurationManager()
configuaration = object.get_data_ingestion_config()
obj = DataIngestion(config= configuaration)
obj.set_kaggle_credentials()
obj.download_dataset()
obj.extract_zip()
obj.verify_data()

[ 2026-05-18 20:06:50,215 : INFO : main : YAML file: config\config.yaml Loaded Successfully ]
[ 2026-05-18 20:06:50,216 : INFO : main : Create Directory at: artifacts ]
[ 2026-05-18 20:06:50,217 : INFO : main : Create Directory at: artifacts/data_ingestion ]
[ 2026-05-18 20:06:50,218 : INFO : 2730326622 : ✅ Kaggle credentials authenticated successfully ]
[ 2026-05-18 20:06:50,219 : INFO : 2730326622 : Downloading dataset: lakshmi25npathi/imdb-dataset-of-50k-movie-reviews ]
[ 2026-05-18 20:06:50,220 : INFO : 2730326622 : Saving to         : artifacts/data_ingestion ]
Dataset URL: https://www.kaggle.com/datasets/lakshmi25npathi/imdb-dataset-of-50k-movie-reviews
[ 2026-05-18 20:06:51,588 : INFO : 2730326622 : ✅ Dataset downloaded successfully ]
[ 2026-05-18 20:06:51,918 : INFO : 2730326622 : ✅ Extracted zip to: artifacts/data_ingestion/ ]
[ 2026-05-18 20:06:51,918 : INFO : 2730326622 : ✅ File verified    : artifacts\data_ingestion\IMDB Dataset.csv ]
[ 2026-05-18 20:06:51,918 : INFO : 2730